In [2]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import boto3

In [3]:
load_dotenv()

True

In [4]:
minio_client = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("SECRET_ACCESS_KEY"),
    endpoint_url= "http://" + os.getenv("API_ENDPOINT")
)

new_bucket = "landing-zone"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except (minio_client.exceptions.BucketAlreadyExists, minio_client.exceptions.BucketAlreadyOwnedByYou):
    print(f"Bucket '{new_bucket}' already exists")

Bucket 'landing-zone' already exists


In [ ]:
url = 'https://www.gencat.cat/transit/opendata/cameres.xml'
response = requests.get(url)
data = response.text
data

'<?xml version="1.0" encoding="UTF-8"?><wfs:FeatureCollection xmlns="http://www.opengis.net/wfs" xmlns:wfs="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:cite="http://www.opengeospatial.net/cite" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengeospatial.net/cite http://localhost:8080/sct-gis/wfs?service=WFS&amp;version=1.0.0&amp;request=DescribeFeatureType&amp;typeName=cite%3Acameres http://www.opengis.net/wfs http://localhost:8080/sct-gis/schemas/wfs/1.0.0/WFS-basic.xsd"><gml:boundedBy><gml:null>unknown</gml:null></gml:boundedBy><gml:featureMember><cite:cameres fid="cameres.fid-6a6e10fd_19cc2dd7e67_-703a"><cite:geom><gml:Point srsName="http://www.opengis.net/gml/srs/epsg.xml#4326"><gml:coordinates xmlns:gml="http://www.opengis.net/gml" decimal="." cs="," ts=" ">2.1849528,41.45989301</gml:coordinates></gml:Point></cite:geom><cite:carretera>C-58</cite:carretera><cite:municipi>Nus Trinitat</cite:municipi><cite:pk>0.50</

In [9]:
import xml.etree.ElementTree as ET
import pandas as pd

root = ET.fromstring(data)

ns = {
    "gml": "http://www.opengis.net/gml",
    "cite": "http://www.opengeospatial.net/cite"
}

rows = []

for cam in root.findall(".//gml:featureMember/cite:cameres", ns):
    carretera = cam.findtext("cite:carretera", default="", namespaces=ns)
    municipi = cam.findtext("cite:municipi", default="", namespaces=ns)
    pk = cam.findtext("cite:pk", default="", namespaces=ns)
    link = cam.findtext("cite:link", default="", namespaces=ns)
    font = cam.findtext("cite:font", default="", namespaces=ns)

    coords = cam.find(".//gml:coordinates", ns)
    lon, lat = None, None
    if coords is not None and coords.text:
        parts = coords.text.strip().split(",")
        if len(parts) == 2:
            lon, lat = parts[0], parts[1]

    rows.append({
        "carretera": carretera.strip(),
        "municipi": municipi.strip(),
        "pk": pk.strip(),
        "link": link.strip(),
        "font": font.strip(),
        "lon": lon,
        "lat": lat
    })

df = pd.DataFrame(rows)
print(df.head())
print(df[["municipi", "link"]].head(10))

ERROR! Session/line number was not unique in database. History logging moved to new session 188
  carretera                 municipi      pk  \
0      C-58             Nus Trinitat    0.50   
1      C-58        MONTCADA I REIXAC    1.82   
2      C-58                 Ripollet    6.15   
3      C-66                 FORALLAC    5.30   
4      C-31  HOSPITALET DE LLOBREGAT  195.60   

                                                link font         lon  \
0  http://mct.gencat.cat/mct2bo/RenderService?sct...  SCT   2.1849528   
1  http://mct.gencat.cat/mct2bo/RenderService?sct...  SCT  2.17446844   
2  http://mct.gencat.cat/mct2bo/RenderService?sct...  SCT   2.1480753   
3  http://mct.gencat.cat/mct2bo/RenderService?sct...  SCT  3.09302557   
4  http://mct.gencat.cat/mct2bo/RenderService?sct...  SCT  2.10125687   

           lat  
0  41.45989301  
1  41.46672443  
2   41.4983473  
3  41.95173528  
4  41.33995348  
                  municipi                                               l

In [ ]:
df['municipi'].value_counts()
df['carretera'].value_counts()

carretera
A-2                                           15
C-31                                          14
C-58                                          13
B-23                                          11
C-16                                           7
                                              ..
Plaça Cerdà                                    1
CG1 / PK 9+791 (La Portalada) / 860 metres     1
T-11                                           1
B-30                                           1
C-33                                           1
Name: count, Length: 78, dtype: int64

In [ ]:
roads_barcelona_core = [
    "B-10",
    "B-20",
    "Ronda del Mig",
    "B-23",
    "C-31",
    "C-32",
    "C-58"
]

municipis_barcelona_propers = [
    "MERIDIANA",
    "DIAGONAL",
    "GRAN VIA",
    "RONDA LITORAL",
    "RONDA DE DALT",
    "RONDA DEL MIG"
    "BARCELONA",
    "L'HOSPITALET DE LLOBREGAT",
    "HOSPITALET DE LLOBREGAT",
    "BADALONA",
    "SANT ADRIÀ DE BESÒS",
    "SANTA COLOMA DE GRAMENET",
    "ESPLUGUES DE LLOBREGAT",
    "CORNELLÀ DE LLOBREGAT",
    "EL PRAT DE LLOBREGAT",
    "PRAT DE LLOBREGAT",
    "SANT JUST DESVERN",
    "SANT JOAN DESPÍ",
    "SANT FELIU DE LLOBREGAT",
    "SANT BOI DE LLOBREGAT",
    "SANT BOI",
    "MONTGAT",
    "MONTCADA I REIXAC",
    "MONTCADA I REIXAC",
    "CERDANYOLA DEL VALLÈS",
    "SANT CUGAT DEL VALLÈS",
    "SANT CUGAT",
    "RUBÍ",
    "CASTELLDEFELS",
    "GAVÀ",
    "VILADECANS",
    "MOLINS DE REI",
    "EL PAPIOL",
    "PAPIOL",
    "SANT VICENÇ DELS HORTS",
    "SANTA COLOMA DE CERVELLÓ"
]

df['conte'] = (df['carretera'].isin(roads_barcelona_core) & df['municipi'].isin(municipis_barcelona_propers))
df = df[df['conte'] == True]
df

,carretera,municipi,pk,link,font,lon,lat,conte
1,C-58,MONTCADA I REIXAC,1.82,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.17446844,41.46672443,True
4,C-31,HOSPITALET DE LLOBREGAT,195.60,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.10125687,41.33995348,True
14,B-23,BARCELONA,0.00,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.10717197,41.38256781,True
15,B-23,SANT JUST DESVERN,3.11,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.07409722,41.37563461,True
69,B-23,SANT FELIU DE LLOBREGAT,7.38,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.02896365,41.38129628,True
76,C-32,GAVÀ,50.37,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.02120264,41.29896673,True
91,C-58,MONTCADA I REIXAC,1.82,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.17446844,41.46672443,True
98,B-20,BADALONA,21.20,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.24274935,41.46599223,True
99,B-20,MONTGAT,24.65,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.27965398,41.4710549,True
100,B-20,SANT BOI,0.83,http://mct.gencat.cat/mct2bo/RenderService?sct...,SCT,2.07121286,41.33229667,True


In [28]:
df_img = df[df["link"].str.contains(r"\.(gif|jpeg|jpg|png)", case=False, na=False)].copy()
print(df_img[["municipi", "link"]].head(20))

                    municipi  \
1          MONTCADA I REIXAC   
4    HOSPITALET DE LLOBREGAT   
14                 BARCELONA   
15         SANT JUST DESVERN   
69   SANT FELIU DE LLOBREGAT   
76                      GAVÀ   
91         MONTCADA I REIXAC   
98                  BADALONA   
99                   MONTGAT   
100                 SANT BOI   
109   SANT VICENÇ DELS HORTS   
110                   PAPIOL   
111                   PAPIOL   
112      SANT ADRIÀ DE BESÒS   
113                 BADALONA   
115                  MONTGAT   
116            CASTELLDEFELS   
117               VILADECANS   

                                                  link  
1    http://mct.gencat.cat/mct2bo/RenderService?sct...  
4    http://mct.gencat.cat/mct2bo/RenderService?sct...  
14   http://mct.gencat.cat/mct2bo/RenderService?sct...  
15   http://mct.gencat.cat/mct2bo/RenderService?sct...  
69   http://mct.gencat.cat/mct2bo/RenderService?sct...  
76   http://mct.gencat.cat/mct2bo/RenderService?s

/var/folders/x6/h1hq1rf10hn6lybhqt5n942w0000gn/T/ipykernel_11068/2849473804.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_img = df[df["link"].str.contains(r"\.(gif|jpeg|jpg|png)", case=False, na=False)].copy()


In [30]:
import os
import requests
from urllib.parse import urlparse

headers = {
    "User-Agent": "Mozilla/5.0"
}

for i, row in df_img.iterrows():
    url = row["link"].strip()

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()

        # detectem extensió
        parsed = urlparse(url)
        filename_from_url = os.path.basename(parsed.path)

        if not filename_from_url:
            filename_from_url = f"camera_{i}.gif"

        municipi = row["municipi"].replace(" ", "_").replace("/", "_")
        carretera = row["carretera"].replace(" ", "_").replace("/", "_")

        filename = f"{i}_{municipi}_{carretera}_{filename_from_url}"
        
        minio_client.put_object(
            Bucket = new_bucket,
            Key = filename,
            Body = response.content
        )

        print(f"Descarregada: {filename}")

    except Exception as e:
        print(f"Error amb {url}: {e}")

Descarregada: 1_MONTCADA_I_REIXAC_C-58_RenderService
Descarregada: 4_HOSPITALET_DE_LLOBREGAT_C-31_RenderService
Descarregada: 14_BARCELONA_B-23_RenderService
Descarregada: 15_SANT_JUST_DESVERN_B-23_RenderService
Descarregada: 69_SANT_FELIU_DE_LLOBREGAT_B-23_RenderService
Descarregada: 76_GAVÀ_C-32_RenderService
Descarregada: 91_MONTCADA_I_REIXAC_C-58_RenderService
Descarregada: 98_BADALONA_B-20_RenderService
Descarregada: 99_MONTGAT_B-20_RenderService
Descarregada: 100_SANT_BOI_B-20_RenderService
Descarregada: 109_SANT_VICENÇ_DELS_HORTS_B-23_RenderService
Descarregada: 110_PAPIOL_B-23_RenderService
Descarregada: 111_PAPIOL_B-23_RenderService
Descarregada: 112_SANT_ADRIÀ_DE_BESÒS_C-31_RenderService
Descarregada: 113_BADALONA_C-31_RenderService
Descarregada: 115_MONTGAT_C-31_RenderService
Descarregada: 116_CASTELLDEFELS_C-31_RenderService
Descarregada: 117_VILADECANS_C-31_RenderService
